# Entity Identification: Deterministic UUIDs and Name Normalization

This notebook demonstrates `siege_utilities.identifiers` — the package for
generating reproducible UUIDs from entity attributes and normalizing names
for fuzzy matching.

## Why deterministic UUIDs?
When the same entity appears in multiple data sources with different IDs,
we need a stable identifier derived from the entity's attributes. A UUID5
generated from a namespace + seed string is deterministic: same inputs
always produce the same UUID.

## 1. Namespace Hierarchies

In [1]:
from siege_utilities.identifiers.namespaces import (
    derive_root,
    derive_sub_namespace,
    NAMESPACE_URL,
)

# Create a root namespace for our organization
org_ns = derive_root("siege-analytics.com")
print(f"Org namespace:     {org_ns}")

# Derive sub-namespaces for different entity types
person_ns = derive_sub_namespace(org_ns, "person")
precinct_ns = derive_sub_namespace(org_ns, "precinct")
print(f"Person namespace:   {person_ns}")
print(f"Precinct namespace: {precinct_ns}")

# Same inputs always produce the same namespace
assert derive_root("siege-analytics.com") == org_ns
print("Deterministic: same input → same output")

Org namespace:     375647c7-a717-5607-8597-1deba4d1abdc
Person namespace:   2ec25019-3480-58ae-b836-1b28bd1a08d0
Precinct namespace: 0d5a0e02-ee1a-5a95-a988-879446d99d58
Deterministic: same input → same output


## 2. Deterministic UUID Generation

In [2]:
from siege_utilities.identifiers.uuid_generation import uuid5_from_seed

# Generate UUIDs from entity attributes
uid1 = uuid5_from_seed(person_ns, "Chand, Dheeraj | Austin TX")
uid2 = uuid5_from_seed(person_ns, "Chand, Dheeraj | Austin TX")
uid3 = uuid5_from_seed(person_ns, "Smith, Jane | Chicago IL")

print(f"Same person:      {uid1}")
print(f"Same person (v2): {uid2}")
print(f"Diff person:      {uid3}")
print(f"Deterministic:    {uid1 == uid2}")
print(f"Unique:           {uid1 != uid3}")

Same person:      686c6dce-4e13-5eef-ae8c-ccd04d91d04d
Same person (v2): 686c6dce-4e13-5eef-ae8c-ccd04d91d04d
Diff person:      0b39da08-cc0d-5a35-a9b5-455140534086
Deterministic:    True
Unique:           True


## 3. Name Normalization for Fuzzy Matching

When matching entities across data sources, names need normalization
to handle case, accents, and whitespace differences.

In [3]:
from siege_utilities.identifiers.normalize import normalize_name_v1

# Normalize handles case, accents, whitespace
names = [
    "José García",
    "JOSE GARCIA",
    "  josé   garcía  ",
]
for name in names:
    normalized = normalize_name_v1(name)
    print(f"{name!r:30} → {normalized!r}")

# All three normalize to the same string
results = [normalize_name_v1(n) for n in names]
print(f"All match: {len(set(results)) == 1}")

'José García'                  → 'jose garcia'
'JOSE GARCIA'                  → 'jose garcia'
'  josé   garcía  '            → 'jose garcia'
All match: True


## Putting it together

The typical workflow for entity resolution:
1. Choose a namespace for the entity type
2. Normalize the entity's identifying attributes
3. Generate a deterministic UUID from the normalized seed
4. Use the UUID for cross-source joins

In [4]:
# Full workflow: normalize name → generate UUID
raw_name = "  María  RODRÍGUEZ  "
normalized = normalize_name_v1(raw_name)
entity_id = uuid5_from_seed(person_ns, normalized)
print(f"Raw:        {raw_name!r}")
print(f"Normalized: {normalized!r}")
print(f"UUID:       {entity_id}")

Raw:        '  María  RODRÍGUEZ  '
Normalized: 'maria rodriguez'
UUID:       18648da2-40e2-57ab-b938-879cc8b3e053
